In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.analytic import ProbabilityOfImprovement
import copy

In [2]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelGP/ModelGP_M25.json")
gp_model.get_next_trials(max_trials=1)

{56: {'n_ci': 0.7707851531048115, 'n_it': 0.3966704003415226}}

In [3]:
def SurrogateModelOfReality(n_ci,n_it):
    y_pred = gp_model.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0]
    return np.float64(y_pred)

In [4]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [5]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0, 1])),
        RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0, 1])),
        RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0, 1])),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": ProbabilityOfImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    # Quasirandom Sampling Exercise
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"s1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"s2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"b1", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    X = sampler.three.QuasirandomSampler3D_func(8,Parameters_lis).T

    for array in X:
        s1 = array[0]
        s2 = array[1]
        b1 = array[2]
        n_ci = PredictorsToCaStoichs(s1,b1)
        n_it = PredictorsToIaStoichs(s2,b1)
        my_parameters = {"s1": s1, "s2": s2, "b1": b1}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(n_ci,n_it)})

    for _ in range(7):
        IterationClient = copy.deepcopy(client)
        IterationTrials = {}
        for __ in range(3):
            SampleTrial = IterationClient.get_next_trials(max_trials=1)
            for trial_index, parameters in SampleTrial.items():
                IterationTrials[trial_index]=parameters
                s1 = parameters["s1"]
                s2 = parameters["s2"]
                b1 = parameters["b1"]
                result = IterationClient.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
                raw_data = {metric_name: result}
                IterationClient.complete_trial(trial_index=trial_index, raw_data=raw_data)
        for trial_index, parameters in IterationTrials.items():
            client.attach_trial(parameters=parameters)
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]
            n_ci = PredictorsToCaStoichs(s1,b1)
            n_it = PredictorsToIaStoichs(s2,b1)
            result = SurrogateModelOfReality(n_ci,n_it)
            raw_data = {metric_name: result}
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)

    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(np.array(client.summarize().t1).tolist()[0:27]))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
13.39385231084375

Trial 1 =========================================
14.928752466872837

Trial 2 =========================================
15.358800703257529

Trial 3 =========================================
17.815850358415815

Trial 4 =========================================
16.723538649562972

Trial 5 =========================================
16.947487268055298

Trial 6 =========================================
13.899999619012675

Trial 7 =========================================
18.275765916552643

Trial 8 =========================================
18.08177063154059

Trial 9 =========================================
18.29215896368141

Trial 10 =========================================
15.621415532648431

Trial 11 =========================================
13.393868853693883

Trial 12 =========================================
13.671256146121301

Trial 13 =========================================
13.230975136013516

Trial 14 ==========

/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.13/site-packages/botorch/fit.py:215: OptimizationWarning: `scipy_minimize` terminated with status OptimizationStatus.FAILURE, displaying original message from `scipy.optimize.minimize`: ABNORMAL: 
  result = optimizer(mll, closure=closure, **optimizer_kwargs)


Trial 42 =========================================
13.807073293820308

Trial 43 =========================================
18.212540995019896

Trial 44 =========================================
13.9067251298945

Trial 45 =========================================
13.910474935924844

Trial 46 =========================================
18.162377410861886

Trial 47 =========================================
15.973345431308502

Trial 48 =========================================
13.892336994456242

Trial 49 =========================================
13.500676494160388

Trial 50 =========================================
13.90963096942075

Trial 51 =========================================
13.283482920972157

Trial 52 =========================================
17.698541406375664

Trial 53 =========================================
18.26331624937106

Trial 54 =========================================
16.960016172458154

Trial 55 =========================================
14.367486552038415

Trial 56 =

/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.13/site-packages/botorch/fit.py:215: OptimizationWarning: `scipy_minimize` terminated with status OptimizationStatus.FAILURE, displaying original message from `scipy.optimize.minimize`: ABNORMAL: 
  result = optimizer(mll, closure=closure, **optimizer_kwargs)


Trial 58 =========================================
17.572373057685546

Trial 59 =========================================
13.251374648503669

Trial 60 =========================================
18.250301499269675

Trial 61 =========================================
14.277978191588641

Trial 62 =========================================
13.109064161665426

Trial 63 =========================================
13.868362263587297

Trial 64 =========================================
17.96784968398708

Trial 65 =========================================
13.795893649336286

Trial 66 =========================================
18.290029896983757

Trial 67 =========================================
13.902160612314796

Trial 68 =========================================
18.22666768630154

Trial 69 =========================================
18.211151574844035

Trial 70 =========================================
13.89784295435325

Trial 71 =========================================
16.245499362732232

Trial 72 

In [6]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.292522346491864
Avg = 15.357070125290184
Std = 1.9484296961024474


In [7]:
print(y_max_arr.tolist())

[13.39385231084375, 14.928752466872837, 15.358800703257529, 17.815850358415815, 16.723538649562972, 16.947487268055298, 13.899999619012675, 18.275765916552643, 18.08177063154059, 18.29215896368141, 15.621415532648431, 13.393868853693883, 13.671256146121301, 13.230975136013516, 13.867515331583233, 18.29021905509891, 18.292522346491864, 13.356630037463413, 18.02482829656797, 13.826736702779957, 13.763502911467619, 17.68391935204033, 16.37620522462574, 17.884776142024272, 18.255521436024978, 15.710572522081673, 13.885370103756156, 13.816718385651116, 13.317327270302723, 16.9846600415492, 13.78418131140578, 14.448231636361978, 13.283791069458374, 13.895708017697396, 13.901579118721902, 13.395709002083937, 13.795888190946176, 13.908534147004987, 16.979815562582836, 13.745646924114894, 17.90842282178444, 17.584304510128177, 13.807073293820308, 18.212540995019896, 13.9067251298945, 13.910474935924844, 18.162377410861886, 15.973345431308502, 13.892336994456242, 13.500676494160388, 13.909630969

In [8]:
# filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/SequentialTestswGPModel/DataGenerated/normal_PI_9_27_3.pkl"
# latestdf = pd.DataFrame(y_max_arr)
# pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)

In [9]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_M25/DataGenerated/normal_PI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)

In [10]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_M25/DataGenerated/normal_PI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
print(loadeddf)
# newdf = loadeddf.drop(loadeddf.index, inplace=True)
# pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)
# print(newdf)

             0
0    13.893656
1    13.907002
2    18.252604
3    18.039412
4    13.784970
..         ...
995  13.324318
996  16.652319
997  18.263727
998  13.836408
999  13.882917

[1000 rows x 1 columns]
